In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
paper_4.2.4.2_overview_MDS_klist.py
-----------------------------------
4.2.4.2 クラスタリング設定の概要（本文用）
- MDS 固有値の寄与率バー＋累積線（OH/FP 並列）
- (mode,index) ごとのクラスタ数 k 一覧（OH/FP）

入力想定:
  ROOT/
    ├─ OH/   or  STEP3_new_MDS_YYYYmmdd/OH/   ← MDS_eigen_contributions_OH_*.csv がこの下にある想定
    ├─ FP/   or  STEP3_new_MDS_YYYYmmdd/FP/   ← MDS_eigen_contributions_FP_*.csv
    ├─ figs_OH/ClusterAssign_(top3|cumeig)_(index)_OH_*.csv
    └─ figs_FP/ClusterAssign_(top3|cumeig)_(index)_FP_*.csv

出力:
  ROOT/paper_4.2.4.2_overview/
    ├─ main_text/
    │   ├─ Fig_4.2.4.2_MDS-eigen_OH-FP.(png|pdf)
    │   └─ Table_4.2.4.2_k-by-index.csv
    └─ analysis_csv/  # 予備（読み取りのログや中間を入れたい場合に拡張可）
"""

from pathlib import Path
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# ========= ユーザー設定 =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")

# MDS出力の探索候補（上から順に存在を探す）
# 例: ROOT/STEP3_new_MDS_20250904/OH/MDS_eigen_contributions_OH_*.csv
MDS_DIR_CANDIDATES = [
    ROOT / "STEP3_new_MDS_20250904",
    ROOT / "STEP3_new_MDS",         # 汎用
    ROOT                             # 直下に OH/FP がある場合
]

# ClusterAssign の所在
FIGS_DIR = {"OH": ROOT / "figs_OH", "FP": ROOT / "figs_FP"}

# 出力（4.2.4.2 本文用）
OUT_ROOT     = ROOT / "paper_4.2.4.2_overview"
OUT_MAIN     = OUT_ROOT / "main_text"
OUT_ANALYSIS = OUT_ROOT / "analysis_csv"
for d in [OUT_ROOT, OUT_MAIN, OUT_ANALYSIS]:
    d.mkdir(parents=True, exist_ok=True)

# 解析対象
DATASETS = ["OH","FP"]
DIMS     = ["top3","cumeig"]
INDICES  = ["silhouette","dunn","gap","ch","db","ptbiserial"]

DPI = 300

# ========= フォント（日本語フォールバック） =========
def set_font():
    cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo"]
    have = set()
    for p in fm.findSystemFonts(fontext="ttf"):
        try: have.add(fm.FontProperties(fname=p).get_name())
        except: pass
    for w in cand:
        if any(w.lower() in h.lower() for h in have):
            matplotlib.rcParams["font.family"] = w
            break
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams.update({"font.size":10, "axes.titlesize":12, "axes.labelsize":10, "legend.fontsize":9})

# ========= ユーティリティ =========
def read_csv_quiet(path: Path) -> pd.DataFrame | None:
    if not path or not path.exists(): return None
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return None

def latest_file(dir_path: Path, pattern_glob: str) -> Path | None:
    cands = sorted(dir_path.glob(pattern_glob), key=os.path.getmtime, reverse=True)
    return cands[0] if cands else None

# ========= MDS eigen 読み込み =========
def find_mds_dir_for(ds: str) -> Path | None:
    # 候補の中から ds サブフォルダを順に探索
    for base in MDS_DIR_CANDIDATES:
        for sub in [base / ds, base]:  # base/ds → base の順に
            if sub.exists():
                # 目当てのファイルがあるか軽く確認
                p = latest_file(sub, f"MDS_eigen_contributions_{ds}_*.csv")
                if p is not None:
                    return sub
    return None

def load_mds_eigen(ds: str) -> pd.DataFrame | None:
    mdir = find_mds_dir_for(ds)
    if mdir is None:
        print(f"[WARN] MDS dir not found for {ds}.")
        return None
    f = latest_file(mdir, f"MDS_eigen_contributions_{ds}_*.csv")
    if f is None:
        print(f"[WARN] MDS_eigen_contributions not found in {mdir} for {ds}.")
        return None
    df = read_csv_quiet(f)
    if df is None:
        print(f"[WARN] Failed to read: {f}")
    return df

# ========= k一覧（ClusterAssign_*） =========
FN_MAIN = re.compile(r"^ClusterAssign_(top3|cumeig)_(silhouette|dunn|gap|ch|db|ptbiserial)_(OH|FP)_.*\.csv$")
def summarize_k_from_assign(figs_dir: Path, ds: str) -> pd.DataFrame:
    rows = []
    for mode in DIMS:
        for ix in INDICES:
            # 最新ファイルを拾う
            matches = sorted(figs_dir.glob(f"ClusterAssign_{mode}_{ix}_{ds}_*.csv"), key=os.path.getmtime, reverse=True)
            if not matches:
                rows.append([ds, mode, ix, np.nan, None]); 
                continue
            f = matches[0]
            df = read_csv_quiet(f)
            if df is None:
                rows.append([ds, mode, ix, np.nan, f.name]); 
                continue
            if {"Variable","Cluster"}.issubset(df.columns):
                k = pd.Series(df["Cluster"]).nunique(dropna=True)
            elif df.shape[1] == 1:
                k = pd.Series(df.iloc[:,0]).nunique(dropna=True)
            else:
                k = np.nan
            rows.append([ds, mode, ix, k, f.name])
    return pd.DataFrame(rows, columns=["Dataset","mode","index","k","source_file"])

# ========= 図作成 =========
def plot_mds_eigen_ohfp(eig_oh: pd.DataFrame | None, eig_fp: pd.DataFrame | None, out_png: Path, out_pdf: Path):
    set_font()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=DPI, sharey=True)

    def _one(ax, df: pd.DataFrame | None, title: str):
        if df is None or ("Share_pos" not in df.columns and "Share" not in df.columns):
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
            ax.set_axis_off()
            return
        # カラム名のゆらぎに対応
        share = df.get("Share_pos", df.get("Share", None))
        cshare = df.get("CumShare_pos", df.get("CumShare", None))
        if share is None:
            ax.text(0.5, 0.5, "No 'Share' column", ha="center", va="center"); ax.set_axis_off(); return
        s = pd.Series(share).fillna(0).values
        c = pd.Series(cshare).ffill().fillna(0).values if cshare is not None else np.cumsum(s)
        # 正の固有値範囲だけ使う前提（0より大きいもの）
        mask = s > 0
        dims = np.arange(1, mask.sum()+1)
        if mask.sum() == 0:
            ax.text(0.5, 0.5, "No positive eigenvalues", ha="center", va="center"); ax.set_axis_off(); return
        ax.bar(dims, s[mask], alpha=0.9)
        ax.plot(dims, c[mask], marker="o")
        ax.set_ylim(0, 1.05)
        ax.set_xlabel("Dimension")
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)

    _one(axes[0], eig_oh, "OH (with material names)")
    _one(axes[1], eig_fp, "FP (fragment presence)")
    axes[0].set_ylabel("Share / Cumulative share (positive eigenvalues)")
    fig.suptitle("Fig. 4.2.4.2: MDS eigenvalue shares and cumulative shares (OH vs FP)")
    fig.tight_layout(rect=[0,0,1,0.94])

    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

# ========= メイン =========
def main():
    # 1) MDS eigen 読み込み
    eig_oh = load_mds_eigen("OH")
    eig_fp = load_mds_eigen("FP")

    # 2) 図：MDS 固有値（本文）
    png = OUT_MAIN / "Fig_4.2.4.2_MDS-eigen_OH-FP.png"
    pdf = OUT_MAIN / "Fig_4.2.4.2_MDS-eigen_OH-FP.pdf"
    plot_mds_eigen_ohfp(eig_oh, eig_fp, png, pdf)
    print(f"[MAIN] Wrote: {png.name}, {pdf.name}")

    # 3) k一覧表：OH/FP 両方（本文）
    rows = []
    for ds in DATASETS:
        tab = summarize_k_from_assign(FIGS_DIR[ds], ds)
        rows.append(tab)
    if rows:
        ktab = pd.concat(rows, ignore_index=True).sort_values(["Dataset","mode","index"])
        out_csv = OUT_MAIN / "Table_4.2.4.2_k-by-index.csv"
        ktab.to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"[MAIN] Wrote: {out_csv.name}")
    else:
        print("[WARN] No ClusterAssign files for k-table.")

    print("\n✅ Done: 4.2.4.2 outputs →", OUT_ROOT)

if __name__ == "__main__":
    main()

[MAIN] Wrote: Fig_4.2.4.2_MDS-eigen_OH-FP.png, Fig_4.2.4.2_MDS-eigen_OH-FP.pdf
[MAIN] Wrote: Table_4.2.4.2_k-by-index.csv

✅ Done: 4.2.4.2 outputs → /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.2_overview
